# Workshop: Implement a Multi-Head Self-Attention Block

* This notebook is prepared with the help of chatGPT.

This notebook is a hands-on practice for writing the **attention block** used in Transformer backbones (including Diffusion Transformers / DiT).

## Goals
1. Implement **scaled dot-product attention**.
2. Build a **multi-head self-attention** module.
3. Verify correctness by matching **PyTorch's** `nn.MultiheadAttention` output (with identical weights).

### What you'll implement
- `scaled_dot_product_attention(...)`
- `MultiHeadSelfAttention.forward(...)`

### Things we need to pay attention to:
- Understand shapes (input, W_Q, W_K, W_V, etc)
- Attention weights sum to 1 (use softmax)
- Numerical equivalence to `torch.nn.MultiheadAttention`
- Gradient check vs. `torch.nn.MultiheadAttention`
- (optional) Causal masking (no information leak from the future)


In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=5, sci_mode=False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cpu


## Part A — Scaled Dot-Product Attention

Recall (per head):
$$
\mathrm{Attn}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d}} + M\right)V
$$

- `Q, K, V` shapes: **(B, H, N, d)**; `B` - Batch size; `H` - number of heads; `N` - input length; `d` - per head dimension.
- `M` (optional) is a broadcastable mask added to logits (use `-inf` for disallowed positions); can be used for causal masking.
- Return both the output and the attention weights.


In [3]:
def scaled_dot_product_attention(Q: torch.Tensor,
                                 K: torch.Tensor,
                                 V: torch.Tensor,
                                 attn_mask: torch.Tensor | None = None,
                                 eps: float = 1e-9):
    """Scaled dot-product attention.

    Args:
        Q, K, V: (B, H, N, d)
        attn_mask: optional additive mask broadcastable to (B, H, N, N)
                   Use 0 for allowed, -inf for disallowed.
    Returns:
        out: (B, H, N, d)
        attn: (B, H, N, N)
    """
    # TODO: implement
    # 1) logits = Q @ K^T / sqrt(d)
    # 2) add attn_mask if provided
    # 3) attn = softmax(logits, dim=-1)
    # 4) out = attn @ V
    
    d = Q.shape[-1]
    # (B,H,N,d) @ (B,H,d,N) -> (B,H,N,N)
    logits = (Q @ K.transpose(-2, -1)) / math.sqrt(d)
    if attn_mask is not None:
        logits = logits + attn_mask  # broadcast to (B,H,N,N)
    attn = torch.softmax(logits, dim=-1)
    out = attn @ V  # (B,H,N,N) @ (B,H,N,d) -> (B,H,N,d)
    return out, attn


## Part B — Multi-Head Self-Attention Module

We will implement a self-attention block.

### Specs
- Input: `x` of shape **(B, N, D)**
- Parameters:
  - A single linear layer for `qkv` of shape **D → 3D** (i.e., combined projection)
  - Output projection: **D → D**
- Split into `H` heads with head dim `d = D/H`
- Support an optional additive `attn_mask` (broadcastable to (B, H, N, N))


In [4]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, bias: bool = True):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=bias)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=bias)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor | None = None, return_attn: bool = False):
        """Self-attention.

        Args:
            x: (B, N, D)
            attn_mask: additive mask broadcastable to (B, H, N, N)
            return_attn: if True, return attention weights too
        Returns:
            y: (B, N, D)
            (optional) attn: (B, H, N, N)
        """
        # TODO:
        # 1) project to qkv: (B, N, 3D)
        # 2) split into q,k,v each (B, N, D)
        # 3) reshape to heads: (B, H, N, d)
        # 4) apply scaled_dot_product_attention
        # 5) merge heads: (B, N, D)
        # 6) output projection

        B, N, D = x.shape
        # (B,N,3D)
        qkv = self.qkv(x)
        # split -> each (B,N,D)
        q, k, v = qkv.chunk(3, dim=-1)

        # reshape to heads: (B,H,N,d)
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        # attention
        out, attn = scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)

        # merge heads: (B,N,D)
        out = out.transpose(1, 2).contiguous().view(B, N, D)

        # output projection
        y = self.proj(out)
        if return_attn:
            return y, attn
        return y


## Utilities for Testing

- `make_causal_mask(N)` creates a **lower-triangular** attention mask (no attending to future tokens).
  
M =
\begin{bmatrix}
0 & -\infty & -\infty & -\infty & -\infty \\
0 & 0 & -\infty & -\infty & -\infty \\
0 & 0 & 0 & -\infty & -\infty \\
0 & 0 & 0 & 0 & -\infty \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}


- `copy_weights_to_torch_mha(...)` copies weights so our implementation and PyTorch's MHA do the *exact same computation*.


In [5]:
def make_causal_mask(N: int, device=None):
    """Additive mask of shape (1, 1, N, N): 0 on allowed, -inf on disallowed."""
    device = device or "cpu"
    # allow i attending to j only if j <= i
    mask = torch.triu(torch.ones(N, N, device=device), diagonal=1).bool()  # True above diag
    additive = torch.zeros(N, N, device=device)
    additive = additive.masked_fill(mask, float("-inf"))
    return additive[None, None, :, :]


def copy_weights_to_torch_mha(ours: MultiHeadSelfAttention, torch_mha: nn.MultiheadAttention):
    """Copy weights so outputs match.

    PyTorch nn.MultiheadAttention uses:
      - in_proj_weight: (3D, D)
      - in_proj_bias: (3D,)
      - out_proj.weight: (D, D)
      - out_proj.bias: (D,)
    Our Linear layers have weight (out, in) as well.
    """
    with torch.no_grad():
        torch_mha.in_proj_weight.copy_(ours.qkv.weight)
        if torch_mha.in_proj_bias is not None:
            torch_mha.in_proj_bias.copy_(ours.qkv.bias)
        torch_mha.out_proj.weight.copy_(ours.proj.weight)
        if torch_mha.out_proj.bias is not None:
            torch_mha.out_proj.bias.copy_(ours.proj.bias)


## Tests (should pass once you finish the TODOs)

Run the cells below after implementing the TODOs.


In [6]:
def test_shapes_and_simplex():
    torch.manual_seed(0)
    B, N, D, H = 2, 7, 32, 4
    x = torch.randn(B, N, D, device=device)
    attn = MultiHeadSelfAttention(D, H).to(device)
    y, A = attn(x, return_attn=True)
    assert y.shape == (B, N, D)
    assert A.shape == (B, H, N, N)
    # attention weights should sum to 1 over last dim
    s = A.sum(dim=-1)
    assert torch.allclose(s, torch.ones_like(s), atol=1e-5), (s.min().item(), s.max().item())
    print("✓ test_shapes_and_simplex passed")


def test_equivalence_to_torch_mha():
    torch.manual_seed(0)
    B, N, D, H = 2, 11, 64, 8
    x = torch.randn(B, N, D, device=device)

    ours = MultiHeadSelfAttention(D, H).to(device)
    torch_mha = nn.MultiheadAttention(embed_dim=D, num_heads=H, dropout=0.0, bias=True, batch_first=True).to(device)
    copy_weights_to_torch_mha(ours, torch_mha)

    y_ours = ours(x)
    y_torch, _ = torch_mha(x, x, x, need_weights=False)
    max_err = (y_ours - y_torch).abs().max().item()
    assert torch.allclose(y_ours, y_torch, atol=1e-5, rtol=1e-5), max_err
    print("✓ test_equivalence_to_torch_mha passed (max_err=%.2e)" % max_err)


def test_causal_mask_no_future_leak():
    torch.manual_seed(0)
    B, N, D, H = 1, 10, 48, 6
    x = torch.randn(B, N, D, device=device)
    attn = MultiHeadSelfAttention(D, H).to(device)
    mask = make_causal_mask(N, device=device)

    y1 = attn(x, attn_mask=mask)
    x2 = x.clone()
    # perturb future tokens heavily; earlier outputs should not change
    x2[:, 7:, :] += 100.0 * torch.randn_like(x2[:, 7:, :])
    y2 = attn(x2, attn_mask=mask)

    # positions < 7 should be identical (no access to tokens 7..)
    max_err = (y1[:, :7, :] - y2[:, :7, :]).abs().max().item()
    assert torch.allclose(y1[:, :7, :], y2[:, :7, :], atol=1e-5, rtol=1e-5), max_err
    print("✓ test_causal_mask_no_future_leak passed (max_err=%.2e)" % max_err)


def test_gradients_match_torch_mha():
    torch.manual_seed(0)
    B, N, D, H = 2, 9, 32, 4
    x = torch.randn(B, N, D, device=device, requires_grad=True)
    x2 = x.clone().detach().requires_grad_(True)

    ours = MultiHeadSelfAttention(D, H).to(device)
    torch_mha = nn.MultiheadAttention(embed_dim=D, num_heads=H, dropout=0.0, bias=True, batch_first=True).to(device)
    copy_weights_to_torch_mha(ours, torch_mha)

    y_ours = ours(x)
    y_torch, _ = torch_mha(x2, x2, x2, need_weights=False)

    loss1 = (y_ours ** 2).mean()
    loss2 = (y_torch ** 2).mean()
    loss1.backward()
    loss2.backward()

    max_err = (x.grad - x2.grad).abs().max().item()
    assert torch.allclose(x.grad, x2.grad, atol=1e-5, rtol=1e-5), max_err
    print("✓ test_gradients_match_torch_mha passed (max_err=%.2e)" % max_err)


# Run all tests
def run_all_tests():
    test_shapes_and_simplex()
    test_equivalence_to_torch_mha()
    test_causal_mask_no_future_leak()
    test_gradients_match_torch_mha()
    print("\nAll tests passed! ✅")


## Once you've implemented the TODOs, run:

```python
run_all_tests()
```


In [8]:
# Uncomment after finishing TODOs:
run_all_tests()


✓ test_shapes_and_simplex passed
✓ test_equivalence_to_torch_mha passed (max_err=8.94e-08)
✓ test_causal_mask_no_future_leak passed (max_err=0.00e+00)
✓ test_gradients_match_torch_mha passed (max_err=1.16e-10)

All tests passed! ✅
